# Quantum Oracle Sketching (QOS) — Quick Start

This notebook demonstrates the core QOS operations: state sketching, oracle sketching, and QSVT transforms.


In [ ]:
import jax
import jax.numpy as jnp
from jax import random

# QOS imports
from qos.core.state_sketch import q_state_sketch_flat, q_state_sketch
from qos.core.oracle_sketch import (
    q_oracle_sketch_boolean,
    q_oracle_sketch_matrix_element,
    q_oracle_sketch_matrix_row_index,
)
from qos.primitives.amplification import amplitude_amplification
from qos.utils.numerical import random_flat_vector, random_unit_vector, random_sparse_matrix

key = random.PRNGKey(0)


## 1. Flat Vector State Sketching

For vectors with entries ±1, QOS constructs the state using a single ancilla qubit.

In [ ]:
N = 1024
key, subkey = random.split(key)
flat_vector = random_flat_vector(subkey, N)

state, num_samples = q_state_sketch_flat(flat_vector, unit_num_samples=1_000_000)
error = jnp.linalg.norm(state - flat_vector / jnp.sqrt(N))
print(f"Norm error: {float(error):.3e} | Samples: {num_samples}")


## 2. General Vector State Sketching

For arbitrary real vectors, QOS uses a Walsh-Hadamard randomization followed by LCU + QSVT arcsin inversion.

In [ ]:
N = 128
key, subkey = random.split(key)
vector = random_unit_vector(subkey, N)

state, total_samples = q_state_sketch(vector, key, unit_num_samples=100_000, degree=20)
fidelity = jnp.abs(jnp.vdot(vector, state / jnp.linalg.norm(state)))**2
print(f"Fidelity: {float(fidelity):.4f} | Total samples: {total_samples}")


## 3. Boolean Phase Oracle Sketching

Construct the phase oracle $\vert x \rangle \mapsto (-1)^{f(x)} \vert x \rangle$ from random truth-table queries.

In [ ]:
N = 1000
key, subkey = random.split(key)
f = random.randint(subkey, (N,), minval=0, maxval=2)

diag, num_samples = q_oracle_sketch_boolean(f, unit_num_samples=10_000_000)
target = jnp.exp(1j * jnp.pi * f)
max_error = float(jnp.max(jnp.abs(diag - target)))
print(f"Max entrywise error: {max_error:.3e} | Samples: {num_samples}")


## 4. Sparse Matrix Element Oracle Sketching

Encode the oracle $\vert i \rangle\vert j \rangle \mapsto A_{ij} \vert i \rangle\vert j \rangle$.

In [ ]:
N1, N2 = 100, 1000
nnz = 300
key, subkey = random.split(key)
A = random_sparse_matrix(subkey, (N1, N2), nnz)

oracle_diag, num_samples = q_oracle_sketch_matrix_element(A, unit_num_samples=1_000_000)
target = A.reshape(N1 * N2)
max_error = float(jnp.max(jnp.abs(oracle_diag - target)))
print(f"Max entrywise error: {max_error:.3e} | Samples: {num_samples}")


## 5. Sparse Matrix Row-Index Oracle Sketching

Encode the oracle that returns the column index of the k-th non-zero element in each row.

In [ ]:
dim1, dim2 = 20, 50
nnz = 100
key, subkey = random.split(key)
A = random_sparse_matrix(subkey, (dim1, dim2), nnz)

index_oracle, num_samples = q_oracle_sketch_matrix_row_index(A, unit_num_samples=10_000_000)
print(f"Oracle shape: {index_oracle.shape} | Samples: {num_samples}")

# Verify first few rows
for i in range(min(3, dim1)):
    pred = jnp.argmax(index_oracle[i, 0])
    nz = jnp.nonzero(A[i])[0]
    print(f"Row {i}: predicted={pred}, actual={nz[0] if len(nz) else 'N/A'}")


## 6. Amplitude Amplification

Boost a low-norm unnormalized state to near-unit norm using QSVT.

In [ ]:
dim = 100
key, subkey = random.split(key)
v = random_unit_vector(subkey, dim) * 0.3  # low norm

amplified = amplitude_amplification(v, degree=51, target_norm=0.99)
print(f"Original norm: {float(jnp.linalg.norm(v)):.3f}")
print(f"Amplified norm: {float(jnp.linalg.norm(amplified)):.4f}")
print(f"Direction error: {float(jnp.linalg.norm(v/jnp.linalg.norm(v) - amplified/jnp.linalg.norm(amplified))):.3e}")


## Next Steps

- Run the full synthetic benchmark: `python -m qos.experiments.benchmark`
- Explore real-dataset experiments in `examples/real_datasets/`
- Read the theory document at `docs/theory.md`
